<a href="https://colab.research.google.com/github/Tokoloshe714/26COA122CW/blob/Searching-for-tools/menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [91]:
"""Most of the following i got from AI as i did not know how to remove the sample data folder that appears automatically every new runtime. I need to remove the file as I am not permitted more than 10 files for submition."""

def remove_colab_sample_data():
    import shutil
    shutil.rmtree("/content/sample_data/", ignore_errors=True)

In [92]:
def search_setup():

    global tool_info_list

    # Define a list[dict{}]. Better than many r/w to .txt file
    tool_info_list = []

    try:
        with open("Tool_Info.txt", "r") as file:
            for line in file:

                line = line.strip()

                if not line: # accounts for empty lines
                    continue

                # copies each attribute into a list
                line_info = [att.strip() for att in line.split(",")]

                if len(line_info) == 5:
                    tool_info_list.append({
                        "ID": line_info[0],
                        "Name": line_info[1],
                        "Size": line_info[2],
                        "Purchase Price": line_info[3],
                        "Purchase Date": line_info[4],
                    })

    except error as e:
        print(f"There was an error: {e}")

    return tool_info_list


#toolSearch

def toolSearch(search_string: str,attribute: str, limit: int= 10):

    if not search_string.strip(): # Empty search box case
        return tool_info_list[:limit]
    else:

        option = [str(tool[attribute]) for tool in tool_info_list]

        # search for all items in tool_info_list according to options
        sorted_matches = process.extract(
            search_string,
            option,
            scorer=fuzz.WRatio, # method for string comparison
            limit=limit
        )

        results = []
        for match_test, score, index in sorted_matches:
            tool = tool_info_list[index].copy()
            tool["Score"] = score
            results.append(tool)
        # return a sorted list of all valid tools in the correct TYPE!!!
        return results



def if_new_query(change, tools_found):

    with tools_found:

        clear_output(wait=True)
        query = search_box.value
        attribute = attribute_dropdown.value
        results = toolSearch(query, attribute)

        if not results:
            print("No matches found.")
        else:
            import pandas as pd
            display(pd.DataFrame(results))

In [93]:
#menu

# AI helped with the following
# • Explain how to setup the CoLab to Github repository connection.
# • Explain how to create and manage projects in Github


# installation and imports
!pip install -q rapidfuzz ipywidgets

from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
from rapidfuzz import process, fuzz
from os import error
from pathlib import Path


def upload_files(): # Upload all files needed with error handling

    files_needed = [
        "Bookings.txt",
        "feedbackManager.txt",
        "Rental_History.txt",
        "Subscription_Info.txt",
        "subscriptionManager.txt",
        "Tool_Feedback.txt",
        "Tool_Info.txt"
    ]

    # Check for missing files
    missing_f = [f for f in files_needed if not Path(f"/content/{f}").exists()]

    if not missing_f:
        print("All required files are already present. Skipping upload.")
        return

    print("Missing files:", missing_f)
    print("Please upload the missing files...")

    try:
        uploaded = files.upload()

        if not uploaded:
            print("Upload was cancelled or no files were selected.")
        else:
            for filename in uploaded.keys():
                print(f"File {filename} uploaded successfully.")

    except Exception as e:
        print(f"Error {e} occurred during the upload.")
        print("Please check the file name and location and try again.")


def setup():    #is first thing to run and is only run once

    remove_colab_sample_data()
    upload_files()
    search_setup()

def manager_login():
    return True

def menu():

    global search_box, attribute_dropdown

    manager_login()

    tool_attributes = ["ID", "Name", "Size", "Purchase Price",  "Purchase Date"]
    search_box = widgets.Text(
        value="",
        placeholder="Type to search...",
        description="Search:",
        continuous_update=True, # updates live while typing
        layout=widgets.Layout(width="400px")
    )

    attribute_dropdown = widgets.Dropdown(
        options=tool_attributes,
        value="Name",
        description="Search in:",
        layout=widgets.Layout(width="250px")
    )

    tools_found = widgets.Output()
    # globals ar'nt working. changed approach to use lambda's
    search_box.observe(lambda change: if_new_query(change, tools_found),    names="value")
    attribute_dropdown.observe(lambda change: if_new_query(change, tools_found), names="value")
    user_interface = widgets.VBox([
        widgets.HTML("Tool Search"),
        widgets.HBox([search_box, attribute_dropdown]), tools_found
        ])

    display(user_interface)
    if_new_query(None, tools_found= tools_found)


setup()
menu()

Missing files: ['Bookings.txt', 'feedbackManager.txt', 'Rental_History.txt', 'Subscription_Info.txt', 'subscriptionManager.txt', 'Tool_Feedback.txt', 'Tool_Info.txt']
Please upload the missing files...


KeyboardInterrupt: 